# The `#POIs` column — settling it with measurements

The one column that never matched. Four tests, in order of how much they depend on us
being right about anything:

| Test | Needs | Settles |
|---|---|---|
| **A** excess-budget | *their table only* | whether `#POIs` can be a post-filter count at all |
| **B** survivor curve | our check-ins | the hard ceiling, and what threshold their number *does* sit at |
| **C** top venues | our check-ins | makes A concrete on real data |
| **D** radius sweep | POI coordinates | whether `#POIs` is a **region catalogue** — a pre-filter vocabulary |

Test A is the important one: it needs no data, no dataset identification, no bounding boxes.
It runs on the three numbers LLMGPR printed.

Reuses `llmgpr_checkins_A.parquet`. Attach `output3`/`output42` via **+ Add Input → Your Work**.
Test D additionally needs venue coordinates and will fetch `raw_POIs.txt` alone (~20 s download,
0.66 GB extract) if no coordinate table is attached — then it emits one so this never repeats.

## 0. Setup

In [10]:
import os, re, gc, zipfile, subprocess, math, itertools
import pandas as pd, numpy as np

WORK = "/kaggle/working"; os.makedirs(WORK, exist_ok=True)
CHUNK = 2_000_000

# LLMGPR Table 1, camera-ready (3 cities) and arXiv v1 (NYC only)
TARGET_3CITY = dict(users=7_507,  pois=80_962, cats=436, ck=1_214_631, groups=1_715)
TARGET_NYC   = dict(users=6_078,  pois=63_445, cats=436, ck=  923_856, groups=1_557)
# the other two datasets in the same table -- test A uses them
TABLE1 = {"Foursquare": dict(users=7_507, pois=80_962, ck=1_214_631),
          "Weeplace":   dict(users=4_560, pois=44_194, ck=  623_654),
          "Gowalla":    dict(users=31_751, pois=81_123, ck=  862_502)}
STATED_MIN = 10          # "users and POIs with less than 10 interactions are removed"

CITY_BBOX = {"New York":    dict(lon_min=-74.3,  lon_max=-73.6,  lat_min=40.4, lat_max=41.0),
             "Chicago":     dict(lon_min=-88.0,  lon_max=-87.5,  lat_min=41.6, lat_max=42.1),
             "Los Angeles": dict(lon_min=-118.7, lon_max=-117.6, lat_min=33.6, lat_max=34.4)}
# Yang's own city centres, dataset_TIST2015_Cities.txt
CITY_CENTRE = {"New York": (40.707864, -73.905237), "Chicago": (41.826546, -87.641298),
               "Los Angeles": (34.000002, -118.250001)}

def find(pat, roots=("/kaggle/input", WORK)):
    hits = []
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, _, fns in os.walk(root):
            if "__MACOSX" in dp: continue
            for fn in fns:
                if re.search(pat, fn, re.I) and not fn.startswith("._"):
                    hits.append(os.path.join(dp, fn))
    return sorted(hits)

def save_table(df, stem):
    try:    df.to_parquet(f"{stem}.parquet", index=False); out = f"{stem}.parquet"
    except ImportError: df.to_csv(f"{stem}.csv", index=False); out = f"{stem}.csv"
    print("wrote", out, f"({len(df):,} rows)"); return out

cached = find(r"llmgpr_checkins_A\.(parquet|csv)$")
assert cached, ("no llmgpr_checkins_A found -- attach the output3 or output42 notebook via "
                "+ Add Input -> Your Work")
print("loading", cached[0])
ck = pd.read_parquet(cached[0]) if cached[0].endswith(".parquet") else pd.read_csv(cached[0], dtype=str)
print(f"{len(ck):,} check-ins | {ck['user_id'].nunique():,} users | "
      f"{ck['venue_id'].nunique():,} POIs | {ck['category'].nunique():,} categories")

loading /kaggle/input/notebooks/yosrkharrat/output3/llmgpr_checkins_A.parquet
2,227,756 check-ins | 152,480 users | 237,728 POIs | 436 categories


## Test A — the excess budget

If every retained POI carries at least 10 check-ins, then

    #check-ins  >=  10 x #POIs

and the slack — the **excess budget** `#check-ins - 10 x #POIs` — is the *entire* supply of
check-ins available to every venue above the floor, summed. Divide it by `#POIs` and you get
the mean excess: how far the average venue can sit above 10.

This is arithmetic on their own printed numbers. Nothing here can be wrong about the dataset,
the region, or the source file.

In [11]:
print(f"{'dataset':<12}{'#POIs':>9}{'#check-ins':>12}{'mean deg':>10}"
      f"{'10 x POIs':>12}{'excess':>10}{'excess/POI':>12}")
print("-" * 77)
for name, t in TABLE1.items():
    floor = STATED_MIN * t["pois"]; exc = t["ck"] - floor
    print(f"{name:<12}{t['pois']:>9,}{t['ck']:>12,}{t['ck']/t['pois']:>10.2f}"
          f"{floor:>12,}{exc:>10,}{exc/t['pois']:>12.2f}")
print("-" * 77)

emp = ck["venue_id"].value_counts()
for k in (1, 5, 10):
    s = emp[emp >= k]
    print(f"MEASURED, 3 cities, POIs with >= {k:>2} check-ins: {len(s):>9,} venues, "
          f"mean degree {s.mean():>6.2f}")
print("\nSo a real >=10 cut on this data leaves a mean degree near 37 -- their three datasets")
print("report 10.63 / 14.11 / 15.00. Gowalla's mean excess of 0.63 check-ins per venue is the")
print("clearest: with a floor of 10 it requires almost every venue to sit *exactly* at 10.")

dataset         #POIs  #check-ins  mean deg   10 x POIs    excess  excess/POI
-----------------------------------------------------------------------------
Foursquare     80,962   1,214,631     15.00     809,620   405,011        5.00
Weeplace       44,194     623,654     14.11     441,940   181,714        4.11
Gowalla        81,123     862,502     10.63     811,230    51,272        0.63
-----------------------------------------------------------------------------
MEASURED, 3 cities, POIs with >=  1 check-ins:   237,728 venues, mean degree   9.37
MEASURED, 3 cities, POIs with >=  5 check-ins:    74,203 venues, mean degree  26.23
MEASURED, 3 cities, POIs with >= 10 check-ins:    42,167 venues, mean degree  41.18

So a real >=10 cut on this data leaves a mean degree near 37 -- their three datasets
report 10.63 / 14.11 / 15.00. Gowalla's mean excess of 0.63 check-ins per venue is the
clearest: with a floor of 10 it requires almost every venue to sit *exactly* at 10.


## Test B — the survivor curve

`#POIs(>= k)` on the unfiltered three-city extract. Two things fall out:

1. **The ceiling.** Filtering only ever removes check-ins, so no subset of this data can have
   more POIs clearing `>= 10` than the whole of it does. That number is a hard upper bound on
   any reported `#POIs` under the stated rule.
2. **The effective threshold.** Where their 80,962 (and the arXiv 63,445) actually sit on the
   curve — i.e. what cut would have produced that count, if it is post-filter at all.

In [13]:
def survivor(d, label, target_pois, kmax=30):
    vc = d["venue_id"].value_counts().to_numpy()
    vc.sort(); vc = vc[::-1]
    n1 = len(vc); tot = vc.sum()
    print(f"\n### {label} -- {tot:,} check-ins over {n1:,} venues")
    hdr = f"{'>=k':>5}{'#POIs':>10}{'frac of >=1':>13}{'check-ins kept':>16}{'mean deg':>10}"
    print(hdr); print("-" * len(hdr))
    curve = {}
    for k in list(range(1, 11)) + [12, 15, 20, 25, 30]:
        s = vc[vc >= k]
        if not len(s): break
        curve[k] = len(s)
        print(f"{k:>5}{len(s):>10,}{len(s)/n1:>13.3f}{s.sum():>16,}{s.mean():>10.2f}")
    print("-" * len(hdr))
    ks = sorted(curve)
    ceil10 = curve.get(STATED_MIN, 0)
    print(f"their #POIs = {target_pois:,}")
    print(f"  ceiling at the stated rule: {ceil10:,} venues clear >= {STATED_MIN} here"
          f"  -> theirs is {target_pois / max(ceil10, 1):.2f}x it"
          f"{'  IMPOSSIBLE' if target_pois > ceil10 else ''}")
    if target_pois > curve[1]:
        print(f"  and it exceeds even #POIs(>=1) = {curve[1]:,} -- above the whole catalogue,")
        print(f"  so NO threshold on this region can produce it. The region must differ.")
    else:
        hit = [k for k in ks if curve[k] <= target_pois][0]
        lo = ks[ks.index(hit) - 1] if ks.index(hit) else None
        if lo:
            print(f"  as a post-filter count it would imply a threshold between "
                  f"k={lo} ({curve[lo]:,}) and k={hit} ({curve[hit]:,}), not {STATED_MIN}")
        else:
            print(f"  it sits at or above the k=1 catalogue ({curve[1]:,})")
    return curve

cur3 = survivor(ck, "ALL 3 CITIES vs CIKM camera-ready", TARGET_3CITY["pois"])
cur1 = survivor(ck[ck["city"].eq("New York")], "NEW YORK vs arXiv v1", TARGET_NYC["pois"])


### ALL 3 CITIES vs CIKM camera-ready -- 2,227,756 check-ins over 237,728 venues
  >=k     #POIs  frac of >=1  check-ins kept  mean deg
------------------------------------------------------
    1   237,728        1.000       2,227,756      9.37
    2   144,303        0.607       2,134,331     14.79
    3   108,427        0.456       2,062,579     19.02
    4    87,938        0.370       2,001,112     22.76
    5    74,203        0.312       1,946,172     26.23
    6    64,353        0.271       1,896,922     29.48
    7    56,783        0.239       1,851,502     32.61
    8    50,841        0.214       1,809,908     35.60
    9    46,042        0.194       1,771,516     38.48
   10    42,167        0.177       1,736,641     41.18
   12    35,979        0.151       1,671,854     46.47
   15    29,297        0.123       1,585,637     54.12
   20    22,217        0.093       1,466,687     66.02
   25    17,723        0.075       1,368,566     77.22
   30    14,411        0.061       1,2

## Test C — where the excess budget actually goes

Test A said Gowalla's entire excess budget is 51,272 check-ins across 81,123 venues. Here is
what the head of a real check-in distribution looks like against a budget that size.

In [14]:
vc = ck["venue_id"].value_counts()
top = vc.head(20)
cat = dict(zip(ck["venue_id"], ck["category"]))
print(f"top 20 venues, 3 cities  (excess = check-ins - {STATED_MIN})")
print(f"{'#':>3}{'check-ins':>11}{'excess':>10}{'cum excess':>12}   category")
print("-" * 70)
cum = 0
for i, (v, n) in enumerate(top.items(), 1):
    cum += n - STATED_MIN
    print(f"{i:>3}{n:>11,}{n - STATED_MIN:>10,}{cum:>12,}   {cat.get(v)}")
print("-" * 70)
for name, t in TABLE1.items():
    budget = t["ck"] - STATED_MIN * t["pois"]
    need = np.searchsorted(np.cumsum(top.to_numpy() - STATED_MIN), budget) + 1
    print(f"{name:<12} whole excess budget = {budget:>9,}  "
          + (f"-> exhausted by the top {need} venues alone" if need <= len(top)
             else "-> survives the top 20"))
print("\nThe top handful of venues in any real city consume the entire budget their table")
print("allows for all ~81k venues combined. The column cannot be post-filter.")

top 20 venues, 3 cities  (excess = check-ins - 10)
  #  check-ins    excess  cum excess   category
----------------------------------------------------------------------
  1     19,007    18,997      18,997   Airport
  2     17,831    17,821      36,818   Airport
  3     14,540    14,530      51,348   Airport
  4     10,988    10,978      62,326   Airport
  5      8,759     8,749      71,075   Airport
  6      7,220     7,210      78,285   Train Station
  7      5,648     5,638      83,923   Airport
  8      4,939     4,929      88,852   Plaza
  9      4,874     4,864      93,716   Train Station
 10      3,806     3,796      97,512   Baseball Stadium
 11      3,794     3,784     101,296   Theme Park
 12      3,748     3,738     105,034   Stadium
 13      2,458     2,448     107,482   Electronics Store
 14      2,315     2,305     109,787   Bus Station
 15      2,312     2,302     112,089   Airport
 16      2,242     2,232     114,321   Art Museum
 17      2,190     2,180     116,501   

## Test D — is `#POIs` a region **catalogue**?

The reading that makes their table coherent: `#users` and `#check-ins` are post-filter, while
`#POIs` is the **POI vocabulary of the region** — every venue in it, not only those the retained
users visited. That is not sloppiness so much as a different object, and their own evaluation
needs it: ranking against *"500 unvisited and nearest POIs within the same region"* requires a
catalogue far larger than the retained users' footprint.

Under that reading `#POIs` is a function of **region extent only** — so it is testable. Sweep the
radius around Yang's city centres and ask: is there one radius where the catalogue is 80,962
*and* an activity cut on users inside it yields ~7,507 users holding ~1,214,631 check-ins?

If yes, their pipeline is recovered. If no radius does both at once, the columns come from
different stages and the table is not jointly reproducible.

This needs venue coordinates, which no artifact so far carries — and the 500-candidate sampler
will need them too, so the table it emits is required downstream regardless.

In [15]:
XY = find(r"llmgpr_pois_xy\.(parquet|csv)$")
if XY:
    print("loading", XY[0])
    pois = (pd.read_parquet(XY[0]) if XY[0].endswith(".parquet")
            else pd.read_csv(XY[0], dtype={"venue_id": str}))
else:
    RP = find(r"raw_POIs\.txt$")
    if not RP:
        ZIP = f"{WORK}/dataset_WWW2019.zip"
        url = ("https://drive.usercontent.google.com/download?"
               "id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&export=download&confirm=t")
        print("fetching the zip for raw_POIs.txt only")
        assert subprocess.run(f'curl -L --fail --retry 3 -o "{ZIP}" "{url}"', shell=True).returncode == 0
        with zipfile.ZipFile(ZIP) as z:
            member = [n for n in z.namelist() if re.search(r"raw_POIs\.txt$", n) and "__MACOSX" not in n][0]
            z.extract(member, WORK); print("extracted", member)
        os.remove(ZIP); RP = find(r"raw_POIs\.txt$")
    keep, rows = set(ck["venue_id"].unique()), []
    for chx in pd.read_csv(RP[0], sep="\t", header=None,
                           names=["venue_id", "lat", "lon", "category", "country"],
                           dtype={"venue_id": str, "category": str},
                           on_bad_lines="skip", chunksize=CHUNK):
        rows.append(chx[chx["venue_id"].isin(keep)][["venue_id", "lat", "lon", "category"]])
    pois = pd.concat(rows, ignore_index=True); del rows; gc.collect()
    pois["lat"] = pd.to_numeric(pois["lat"], errors="coerce")
    pois["lon"] = pd.to_numeric(pois["lon"], errors="coerce")
    pois = pois.dropna(subset=["lat", "lon"]).drop_duplicates("venue_id")
    save_table(pois, f"{WORK}/llmgpr_pois_xy")

print(f"{len(pois):,} venues with coordinates "
      f"({len(pois) / ck['venue_id'].nunique():.1%} of the extract's POIs)")

wrote /kaggle/working/llmgpr_pois_xy.parquet (237,728 rows)
237,728 venues with coordinates (100.0% of the extract's POIs)


In [22]:
def haversine_km(lat, lon, lat0, lon0):
    R = 6371.0088
    p, p0 = np.radians(lat), math.radians(lat0)
    dp, dl = p - p0, np.radians(lon - lon0)
    return 2 * R * np.arcsin(np.sqrt(np.sin(dp / 2) ** 2 +
                                     np.cos(p) * math.cos(p0) * np.sin(dl / 2) ** 2))

d = np.full(len(pois), np.inf)
for c, (la, lo) in CITY_CENTRE.items():
    d = np.minimum(d, haversine_km(pois["lat"].to_numpy(), pois["lon"].to_numpy(), la, lo))
pois = pois.assign(km=d)
V2KM = dict(zip(pois["venue_id"], pois["km"]))
ck = ck.assign(km=ck["venue_id"].map(V2KM))
print("check-ins with a distance:", f"{ck['km'].notna().sum():,} / {len(ck):,}")

def match4(got, t):
    return float(np.mean([min(got[k], t[k]) / max(got[k], t[k]) for k in ("users", "pois", "cats", "ck")]))

TU = (1, 2, 3, 5, 10, 20, 30, 40, 50, 60, 75, 100, 125, 150)
print(f"\n{'R km':>6}{'catalogue':>11}{'best Tu':>9}{'users':>9}{'check-ins':>12}"
      f"{'cats':>6}{'ck/user':>9}{'match4':>8}")
print("-" * 70)
best = None
for R in (5, 8, 10, 12, 15, 18, 20, 25, 30, 40):
    reg = ck[ck["km"] <= R]
    if reg.empty: continue
    cat_n = reg["venue_id"].nunique()          # catalogue = venues with >=1 check-in in region
    uc = reg["user_id"].value_counts()
    cand = []
    for Tu in TU:
        sub = reg[reg["user_id"].isin(uc[uc >= Tu].index)]
        if sub.empty: continue
        got = dict(users=sub["user_id"].nunique(), pois=cat_n,
                   cats=sub["category"].nunique(), ck=len(sub))
        cand.append((match4(got, TARGET_3CITY), Tu, got))
    if not cand: continue
    m, Tu, got = max(cand)
    print(f"{R:>6}{cat_n:>11,}{Tu:>9}{got['users']:>9,}{got['ck']:>12,}"
          f"{got['cats']:>6}{got['ck'] / got['users']:>9.1f}{m:>8.3f}")
    if best is None or m > best[0]: best = (m, R, Tu, got)
print("-" * 70)
t = TARGET_3CITY
print(f"{'TARGET':>6}{t['pois']:>11,}{'':>9}{t['users']:>9,}{t['ck']:>12,}"
      f"{t['cats']:>6}{t['ck'] / t['users']:>9.1f}{1.0:>8.3f}")
m, R, Tu, got = best
print(f"\nbest catalogue reading: R = {R} km, users >= {Tu}  -> match {m:.3f}")
for kk, lab in (("users", "users"), ("pois", "POIs (catalogue)"), ("cats", "categories"), ("ck", "check-ins")):
    print(f"    {lab:<20}{got[kk]:>12,} vs {t[kk]:>12,}   {got[kk] / t[kk]:.2f}x")

check-ins with a distance: 2,227,756 / 2,227,756

  R km  catalogue  best Tu    users   check-ins  cats  ck/user  match4
----------------------------------------------------------------------
     5     12,306        3    6,281      78,483   380     12.5   0.481
     8     47,131       10   10,868     372,595   420     34.3   0.636
    10     82,188       30    7,051     574,546   423     81.5   0.842
    12     96,498       30    7,951     668,589   424     84.1   0.827
    15    121,371       40    7,123     750,426   425    105.4   0.802
    18    141,188       40    8,094     870,129   429    107.5   0.800
    20    152,906       40    8,591     936,057   431    109.0   0.791
    25    179,233       50    7,754     992,499   432    128.0   0.807
    30    199,037       50    8,435   1,096,810   432    130.0   0.798
    40    221,023       60    7,437   1,120,511   431    150.7   0.817
----------------------------------------------------------------------
TARGET     80,962          

## Verdict

In [23]:
print("=" * 78)
gow = TABLE1["Gowalla"]
print(f"A  their own table: Gowalla needs {gow['pois']:,} venues at >= {STATED_MIN} check-ins from")
print(f"   {gow['ck']:,} -- mean {gow['ck']/gow['pois']:.2f}, excess {gow['ck']-STATED_MIN*gow['pois']:,}"
      f" across all of them. Not a post-filter count.")
print(f"B  ceiling here: only {cur3.get(10,0):,} of {ck['venue_id'].nunique():,} three-city venues clear >= 10")
print(f"   at all, against their {TARGET_3CITY['pois']:,}. No filter closes that -- filters only remove.")
print(f"C  the top few venues alone exhaust the excess budget their table allows for ~81k.")
print(f"D  catalogue reading scores {m:.3f} at R = {R} km, Tu >= {Tu}.")
print("=" * 78)
print("\nRead D against the 0.923 from the threshold sweep (Tu=60, Tp=3, single pass):")
print("  higher  -> #POIs is a region catalogue; keep the catalogue convention and say so.")
print("  similar -> both readings are equally partial; report ours, footnote theirs.")
print("  lower   -> stay with Tu=60/Tp=3 and treat #POIs as unreproducible.")
print("\nEither way the POI column is closed as a *reporting* question, not a tuning one.")
print("Coordinate table" + (" reused from" if XY else " emitted to") +
      " llmgpr_pois_xy -- the 500-nearest-candidate sampler needs it.")

A  their own table: Gowalla needs 81,123 venues at >= 10 check-ins from
   862,502 -- mean 10.63, excess 51,272 across all of them. Not a post-filter count.
B  ceiling here: only 42,167 of 237,728 three-city venues clear >= 10
   at all, against their 80,962. No filter closes that -- filters only remove.
C  the top few venues alone exhaust the excess budget their table allows for ~81k.
D  catalogue reading scores 0.842 at R = 10 km, Tu >= 30.

Read D against the 0.923 from the threshold sweep (Tu=60, Tp=3, single pass):
  higher  -> #POIs is a region catalogue; keep the catalogue convention and say so.
  similar -> both readings are equally partial; report ours, footnote theirs.
  lower   -> stay with Tu=60/Tp=3 and treat #POIs as unreproducible.

Either way the POI column is closed as a *reporting* question, not a tuning one.
Coordinate table emitted to llmgpr_pois_xy -- the 500-nearest-candidate sampler needs it.
